# TürkResearcher — Stage 2a: Continued Pre-Training

Adapts `Trendyol/Trendyol-LLM-7b-base-v1.0` to Turkish academic register using **740K YÖK Tez + DergiPark abstracts** (~387M tokens, packed into max_seq=2048 chunks).

Output: `hakansabunis/trakad-7b-base` — the foundation model for Stage 2c QLoRA SFT.

**Önce:** Runtime → Change runtime type → **A100 GPU** (Colab Pro+). T4 yetmez, V100/L4 marjinal.

**Akış (~12-20 saat):**
1. Setup + GPU check
2. HF auth + workspace
3. Pull pre-tokenized corpus from HF Hub
4. Load Trendyol-7B base (bf16)
5. Smoke training (200 steps, ~15 dk) — pipeline doğrulama
6. Full training (1 epoch, ~15-18 saat)
7. Push to `hakansabunis/trakad-7b-base`

**Recipe rasyonel:**
- LR=1e-5: düşük — pre-trained Türkçe bilgisini *silmek değil, akademik domain'e kaydırmak* hedefi (catastrophic forgetting'ten kaçınma)
- Effective batch=32 (micro=1 × accum=32): A100 40GB sınırı içinde, gradient stable
- bf16 mixed precision: A100 tensor core'larında hızlı + numeric stable (fp16'dan tercih)
- gradient_checkpointing=True: 7B + 2048 seq için bellek zorunluluğu
- 1 epoch: domain adaptation için yeterli, 2+ overfit riski

## 1. Setup

In [ ]:
!nvidia-smi
!pip install -q --upgrade "transformers>=4.45,<5.0" "datasets>=2.20" "accelerate>=0.34" sentencepiece huggingface_hub pyarrow

## 2. HF auth + workspace

In [ ]:
import os
WORK_DIR = '/content/trakad_pretrain'
os.makedirs(WORK_DIR, exist_ok=True)
print('Working dir:', WORK_DIR)
!df -h /content | tail -1

In [ ]:
from getpass import getpass
from huggingface_hub import login

hf_token = getpass('HF write token: ')
login(hf_token, add_to_git_credential=False)

## 3. Pull pre-tokenized corpus from HF Hub

Source: `hakansabunis/tr-academic-research-agent-index/pretrain_corpus/` — 222,830 chunks of max_seq=2048, fill ratio 84.5%.

In [ ]:
from huggingface_hub import hf_hub_download

TRAIN_PARQUET = hf_hub_download(
    repo_id='hakansabunis/tr-academic-research-agent-index',
    filename='pretrain_corpus/train.parquet',
    repo_type='dataset',
    local_dir=WORK_DIR,
)
VAL_PARQUET = hf_hub_download(
    repo_id='hakansabunis/tr-academic-research-agent-index',
    filename='pretrain_corpus/val.parquet',
    repo_type='dataset',
    local_dir=WORK_DIR,
)
STATS = hf_hub_download(
    repo_id='hakansabunis/tr-academic-research-agent-index',
    filename='pretrain_corpus/_corpus_stats.json',
    repo_type='dataset',
    local_dir=WORK_DIR,
)

import json, os
stats = json.load(open(STATS))
print('Corpus stats:')
for k, v in stats.items():
    print(f'  {k:25s} {v}')
print(f'\nTrain parquet: {os.path.getsize(TRAIN_PARQUET)/1024/1024:.0f} MB')
print(f'Val parquet:   {os.path.getsize(VAL_PARQUET)/1024/1024:.0f} MB')

In [ ]:
from datasets import Dataset
import pandas as pd

train_df = pd.read_parquet(TRAIN_PARQUET)
val_df = pd.read_parquet(VAL_PARQUET)
print(f'Train chunks: {len(train_df):,}')
print(f'Val chunks:   {len(val_df):,}')

# Build HF Datasets. input_ids already a list of ints per row.
train_ds = Dataset.from_pandas(train_df)
val_ds = Dataset.from_pandas(val_df)

# Add labels column = input_ids (standard causal LM convention; Trainer
# shifts internally for next-token prediction)
def add_labels(ex):
    ex['labels'] = ex['input_ids']
    return ex

train_ds = train_ds.map(add_labels, num_proc=4)
val_ds = val_ds.map(add_labels, num_proc=4)
print(f'\nFirst sample length: {len(train_ds[0]["input_ids"])}')
print(f'Last 10 tokens: {train_ds[0]["input_ids"][-10:]}')

## 4. Load Trendyol-7B base (bf16 + gradient checkpointing)

7B params × 2 bytes (bf16) = 14 GB weights. Gradients another 14 GB. Adam states fp32: 28 GB. Activations dominate beyond that; gradient checkpointing trades compute for memory.

In [ ]:
import torch
from transformers import LlamaTokenizer, AutoModelForCausalLM

BASE = 'Trendyol/Trendyol-LLM-7b-base-v1.0'
print(f'Loading tokenizer: {BASE}')
tokenizer = LlamaTokenizer.from_pretrained(BASE)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token  # standard for Llama
print(f'  vocab={tokenizer.vocab_size}, eos={tokenizer.eos_token_id}, pad={tokenizer.pad_token_id}')

print(f'\nLoading model in bf16: {BASE}')
model = AutoModelForCausalLM.from_pretrained(
    BASE,
    torch_dtype=torch.bfloat16,
    device_map='auto',
)
model.gradient_checkpointing_enable()
model.config.use_cache = False  # required when gradient_checkpointing is on
print(f'  params: {sum(p.numel() for p in model.parameters())/1e9:.2f}B')
print(f'  device: {next(model.parameters()).device}')
print(f'  dtype:  {next(model.parameters()).dtype}')

## 5. Smoke training (200 steps, ~15 dk)

Trains for a couple hundred steps so we can confirm: model loads, loss decreases, no OOM, no NaN. Then full training.

In [ ]:
from transformers import TrainingArguments, Trainer, DataCollatorForLanguageModeling

# Pre-tokenized & packed — collator just needs to stack into tensors.
# We disable MLM since this is causal LM.
collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

SMOKE_DIR = f'{WORK_DIR}/smoke_out'
smoke_args = TrainingArguments(
    output_dir=SMOKE_DIR,
    max_steps=200,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=32,  # effective batch 32
    learning_rate=1e-5,
    warmup_ratio=0.03,
    bf16=True,
    logging_steps=10,
    save_strategy='no',
    report_to=[],
    gradient_checkpointing=True,
    optim='adamw_torch',
    lr_scheduler_type='cosine',
)

smoke_trainer = Trainer(
    model=model,
    args=smoke_args,
    train_dataset=train_ds.select(range(min(2000, len(train_ds)))),
    data_collator=collator,
)
smoke_trainer.train()
print('\nSmoke OK — pipeline healthy.')

## 6. Full training (1 epoch, ~15-18 saat)

Resumable: if Colab disconnects mid-run, re-running this cell resumes from latest checkpoint (saved every 1000 steps).

**Eligibility check:** Yalnızca smoke OK çıktıysa burayı çalıştır. Yoksa hataları diagnose et.

In [ ]:
import math, os

OUT_DIR = f'{WORK_DIR}/trakad-7b-base'
EFFECTIVE_BATCH = 32
EPOCHS = 1

# Steps/epoch = chunks / effective_batch
steps_per_epoch = math.ceil(len(train_ds) / EFFECTIVE_BATCH)
print(f'Steps/epoch: {steps_per_epoch:,}')

args = TrainingArguments(
    output_dir=OUT_DIR,
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=EFFECTIVE_BATCH,
    learning_rate=1e-5,
    warmup_ratio=0.03,
    weight_decay=0.01,
    bf16=True,
    logging_steps=20,
    save_strategy='steps',
    save_steps=1000,
    save_total_limit=3,
    eval_strategy='steps',
    eval_steps=1000,
    report_to=[],
    gradient_checkpointing=True,
    optim='adamw_torch',
    lr_scheduler_type='cosine',
    dataloader_num_workers=2,
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    data_collator=collator,
)

# Resume if a checkpoint exists in OUT_DIR (Colab disconnect recovery)
resume = os.path.isdir(OUT_DIR) and any(
    name.startswith('checkpoint-') for name in os.listdir(OUT_DIR)
)
print(f'Resume from checkpoint: {resume}')

import time; t0 = time.time()
trainer.train(resume_from_checkpoint=resume)
dt = time.time() - t0
print(f'\nFull training done in {dt/3600:.2f} hours -> {OUT_DIR}')

# Save tokenizer alongside the model for HF Hub completeness
tokenizer.save_pretrained(OUT_DIR)
print('Tokenizer saved alongside model.')

## 7. Quick PPL sanity check

Evaluate on val set + a few raw academic sample prompts. PPL should be substantially below vanilla Trendyol-7B's PPL on the same val.

In [ ]:
import math
eval_metrics = trainer.evaluate()
print(f'Final val loss: {eval_metrics["eval_loss"]:.4f}')
print(f'Final val PPL:  {math.exp(eval_metrics["eval_loss"]):.2f}')

In [ ]:
# Generate a few Turkish academic continuations to sanity-check the model writes coherent Turkish
prompts = [
    'Bu çalışmada, derin öğrenme yöntemleri kullanılarak Türkçe metinler üzerinde',
    'Yenilenebilir enerji kaynaklarından rüzgar enerjisinin Türkiye\'deki potansiyeli',
    'Akademik araştırmalarda doğal dil işleme teknikleri',
]
model.eval()
for p in prompts:
    inputs = tokenizer(p, return_tensors='pt').to(model.device)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=120, do_sample=True, top_p=0.9, temperature=0.7)
    print(f'\n=== {p}')
    print(tokenizer.decode(out[0], skip_special_tokens=True))

## 8. Push to HF Hub: `hakansabunis/trakad-7b-base`

In [ ]:
from huggingface_hub import HfApi

REPO = 'hakansabunis/trakad-7b-base'
api = HfApi(token=hf_token)
api.create_repo(REPO, repo_type='model', exist_ok=True, private=False, token=hf_token)

print(f'Uploading {OUT_DIR} -> {REPO}')
api.upload_folder(
    folder_path=OUT_DIR,
    repo_id=REPO,
    repo_type='model',
    commit_message='Trakad-7B-base — continued pre-train of Trendyol-LLM-7b-base-v1.0 on 387M tokens of Turkish academic abstracts',
    token=hf_token,
)
print(f'\nDone: https://huggingface.co/{REPO}')